# Rapport TP 1 Multi-agents

## Nathan MEGUIN
## Evahn LE GAL

## Le 23 Octrobre 2023

### Introduction

Le but de ce TP était de simuler un environnement où évoluent des agents à différents états, dans le but de reproduire la propagation d'une épidémie dans une population.

Un individu n'a qu'un seul état parmis entre 4 états distincts :
- Susceptible : l'individu possède le risque d'etre contaminé si il est en présence d'infectés
- Exposé : l'individu a été en contact avec un infecté et il deviendra lui même infecté dans un certain temps
- Infecté : l'individu est susceptible de contaminer les autres autour de lui tant qu'il est infecté
- Rétabli : l'indidu est soigné et possède encore une immunité naturelle au virus,  il ne peut donc plus être contaminé pendant un certain temps

### Le code en C++

Nous avons choisi le C++ comme langage d'implémentation car il faut un langage objet, rapide, proche de la machine, car le nombre de simulations est grand et il faut être le plus efficace possible. De plus, c'était une bonne introduction pour découvrir le langage et ses subtilités dans un projet concret.

Pour le diagramme de classes, nous avons choisi :
- Une énumeration Statut pour nommer les états des individus
- Une classe RandomGenerator pour créer des nombres pseudo-aléatoires. Il implémente le générateur de Mersenne-Twister du C++ pour de meilleurs résultats.
- Une classe Human pour les informations relatives à un individu (son état, son temps dans un état, sa position, etc.)
- Une classe Grid avec le nombre d'infectés par case. Nous avons fait ce choix pour simplifier énormement le calcul des infectés autour d'un individu lors du changement d'état. Il est beaucoup plus aisé de lire 9 valeurs que de parcourir les 20 000 individus et incrémenter pour chaque infecté rencontré. Le seul défaut de cette méthode est qu'il faut mettre à jour à la fois la position d'un individu dans ses attributs, et en même temps incrémenter ou décrémenter les cases correspondantes dans la grille. Cependant, c'est un petit mal pour gagner de bien meilleures performances.
- Une classe Simulation pour traiter toutes les méta-données, telles que le nombre de jours simulés, ou encore les autres instances de classes comme une grille, un tableau d'individus. Cette classe implémente également toutes les méthodes pour lancer une journée de simulation, faire le déplacement d'un individu (et changer la grille) ainsi que mettre son état à jour en fonction du nombre d'infectés autour ou du temps passé dans un certain état. A noter que le tableau d'individus peut etre mélangé puis lu une case après l'autre pour réaliser la simulation d'un individu (dans un ordre aléatoire donc).

![Diagramme de classe](diagrammeClasse.png)

### Le code en Python

Nous utilisons la librairie MathPlotLib de Python pour afficher facilement les graphiques et les sauvegarder. Le dossier "out" contient tous les fichiers .csv d'une simulation (730 jours). Il est possible d'aller lire ces fichiers, d'en extraire les données et ensuite de les représenter sous forme de graphique. Les 3 premiers graphiques générés sont affichés dans le jupyter au moment de l'excecution.

Dans le dossier "graph" se trouve les graphiques des 100 simulations, et lancer le jupyter notebook les mettera à jour. Pour simplement faire la moyenne entre les simulations sans les afficher ni les sauvegarder, il faut passer le champ saveGraph à False dans les fonctions create_graph().

A la fin, un graphique de la moyenne des 100 simulations est affiché et sauvegardé sous un nom spécial. On peut voir notamment que les courbes sont bien plus "lisses" que sur les autres graphiques.

### Conclusion

De ce que nous pouvons voir des résultats, il y a une très forte augmentation du nombre d'infectés (environ 7500 au maximum) dans les premiers jours tandis que le nombre d'individus suscpetibles s'écroule. Puis, le nombre d'infectés diminue rapidement tandis qu'une grosse phase d'individus rétablis se met en place. Puis certains rétablis redeviennent susceptibles (les premiers à s'etre fait contaminés et qui ont une phase de rétablissement courte). Cependant, il n'y a plus assez d'infectés comparé au nombre de rétablis, ce qui limite beaucoup la propagation et qui empeche les susceptibles d'être à nouveau infectés. Ainsi, les niveaux de chaques états ne bougent plus et se stabilisent jusqu'à la fin.

Avec cette simulation, on a ainsi recrée, en bien plus simple, l'évolution d'une vraie épidémie avec ses différentes phases et comment le nombre d'individus dans chaque état influe sur les autres. Ce sont des résultats très interessants, surtout dans le cas d'un TP multi-agents, mais on pourrait étendre les critères ainsi que le déplacement des populations pour cibler des environnements plus réalistes, comme par exmeple le fait que les individus infectés ont plus de chance d'aller sur une case en particulier (l'hôpital) pour se soigner plus vite. Il serait ainsi interessant d'étudier la propagation spaciale de ces populations et de leurs états.

In [ ]:
## Les 100 simulations prennent en moyenne 300 secondes, merci de patienter
!make
!Tp1_multi_agent

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def create_graph(nameGraph, nameFile, fileOut, displayGraph, saveGraph):

    data = np.genfromtxt(nameFile, delimiter=';')

    if saveGraph :
        days = data[:,0]
        susceptible = data[:,1]
        exposed = data[:,2]
        infected = data[:,3]
        recovered = data[:,4]

        plt.plot(days, susceptible, days, exposed, days, infected, days, recovered)
        plt.legend(['Susceptible', 'Exposed', 'Infected', 'Recovered'])
        plt.title(nameGraph)
        plt.xlabel('Nombre de jours')
        plt.ylabel('Nombre d\'individus')

        plt.savefig(fileOut)

        if displayGraph :
            plt.show()

        plt.clf()
    
    return data

In [ ]:
dataAll = np.zeros((730, 5))

In [ ]:
for i in range(1, 100):
    nameGraph = f"Graphique simulation n°{i} épidémie"
    nameFile = f"out/simulation{i}.csv"
    fileOut = f"graphs/graph_simulation{i}.png" 
    dataAll += create_graph(nameGraph, nameFile, fileOut, True, True)

In [ ]:
for i in range(4, 101):
    nameGraph = f"Graphique simulation n°{i} épidémie"
    nameFile = f"out/simulation{i}.csv"
    fileOut = f"graphs/graph_simulation{i}.png"
    dataAll += create_graph(nameGraph, nameFile, fileOut, False, True)

In [ ]:
daysMean = dataAll[:,0]/100
susceptibleMean = dataAll[:,1]/100
exposedMean = dataAll[:,2]/100
infectedMean = dataAll[:,3]/100
recoveredMean = dataAll[:,4]/100

plt.plot(daysMean, susceptibleMean, daysMean, exposedMean, daysMean, infectedMean, daysMean, recoveredMean)
plt.legend(['Susceptible', 'Exposed', 'Infected', 'Recovered'])
plt.title('Graphique moyen des simulations épidémie')
plt.xlabel('Nombre de jours')
plt.ylabel('Nombre d\'individus')
    
plt.savefig('graphs\\graph_simulations_mean.png')

plt.show()